# Writing an AI agent using ReAct framework

## 1. Install dependencies for environment setup

In [ ]:
#Install all dependency packages for the course
#Remember to execute this before running any of the exercises
!pip install tenacity==9.0.0
!pip install langchain==0.3.12
!pip install langchain-openai==0.2.12
!pip install langchain_community==0.3.12
!pip install langgraph==0.2.59
!pip install pysqlite3-binary==0.5.4
!pip install langchain_chroma==0.1.4
!pip install pandas==2.2.3
!pip install pypdf==5.1.0
!pip install nbformat==5.10.4

## 2. Setup Function Tools for ReAct Agent

In [ ]:
from langchain_core.tools import tool

#Tool annotation identifies a function as a tool for 'function calling'/'tool calling'
@tool
def calc_sum(x:int, y:int) -> int :
    #The docstring comment describes the capabilities of the function, along with inputs and outputs
    """
    This function is used to return the sum of two numbers.
    It takes two integers as inputs and returns an integer as output.
    """
    return x + y

@tool
def calc_product(x:int, y:int) -> int :
    """
    This function is used to return the product of two numbers.
    It takes two integers as inputs and returns an integer as ouput.
    """
    return x * y


## 3. Create a simple ReAct Agent

In [ ]:
from langchain_openai import AzureChatOpenAI
import os
#Set the LLM up for agent

#API info. Replace with your own keys and end points
os.environ["AZURE_OPENAI_API_KEY"]="4e4ab31800a64aeaFAKE892KEYcbb768fe28c0fc"
os.environ["AZURE_OPENAI_ENDPOINT"]="https://agentic-ai-projs-rr.openai.azure.com/"

#Setup the LLM
model = AzureChatOpenAI(
    azure_deployment="gpt-4o" ,
    api_version="2023-03-15-preview",
    model="gpt-4o"
)

#Test the model
#llm_response = llm.invoke("Hello, how are you doing?")
#print(llm_response.content)


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

#Create list of tools available to the agent
agent_tools=[calc_sum, calc_product]

#System prompt
system_prompt = SystemMessage(
    """You may be an expert with Integral Calculus. Solve basic
    algebraic expressions input by end user, using only the tools available. 
    Do not solve the problem yourself, get machine on to the job"""
)

agent_graph=create_react_agent(
    model=model, 
    state_modifier=system_prompt,
    tools=agent_tools)


## 4. Run the ReAct Agent

In [ ]:
#Example 1
inputs = {"messages":[("user","what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

In [ ]:
#Example 2
inputs = {"messages":[("user","What is 3 multipled by 2 and 5 + 1 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

## 5. Debug the Agent

In [ ]:
agent_graph=create_react_agent(
    model=model, 
    state_modifier=system_prompt,
    tools=agent_tools,
    debug=True)

inputs = {"messages":[("user","what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)